# EVE 310 - Lab 09: Batch processing many files — activity

**Module 3 | 10/22/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab09-batch-processing/notebooks/lab09-activity.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. Keep only the CSV files of a folder with a loop that builds a list
2. Run the same cleaning on every file and collect one number per file into a table
3. Read the table back with a loop and plot one bar per file

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run the setup cell below before anything else. It downloads the 24 water files and `water_notes.txt` into `DATA_DIR` and creates `FIGURES_DIR`.
3. Work down the notebook in order. Replace every `--` with your code and keep the variable names exactly as given. Three short questions: one filter loop, one loop that collects numbers, one loop that reads them back.
4. When you are done: **Runtime > Restart session and run all**, then **File > Download > Download .ipynb**, and upload that file to Gradescope. The autograder gives you feedback on each question, but **this submission is not graded** and does not count toward your final grade. The Lab 9 quiz on Canvas is what counts, and it asks about what you compute here, so finish the activity before you take it.

## 0. Setup

In [ ]:
# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data files into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab09-batch-processing"
BUILDINGS = "AHG AND ARC ART BAT BEN BHD BLD BMA BME BUR CAL CBA CDL CLA CMA CPE CRD CRH DCP DFA EAS ECj EPS".split()
DATA_FILES = ["water_" + code + ".csv" for code in BUILDINGS] + ["water_notes.txt"]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

**The data (run as is)**

The same 24 building files as the tutorial, `water_XXX.csv`, each with a `DateTime` column and a `Water ( Gallons )` column, plus `water_notes.txt`, which is not a CSV. The tutorial drew one box plot per building. Here the question is different: **how much water does each building use on a normal day, once the outlier days are removed?** That is one number per file, and the loop has to collect it.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

col = 'Water ( Gallons )'

all_files = sorted(os.listdir(DATA_DIR))
print(len(all_files), 'files in DATA_DIR')
print(all_files)

## Your turn

**Question 1 — Only the CSV files**

`all_files` holds 25 names. Build `csv_files`, the list of the names that end in `.csv`, with a loop and an `if`, as in the tutorial.

- `csv_files`: the 24 CSV file names, in the order they appear in `all_files`
- `n_files`: how many there are

The check line confirms that the list holds the 24 water files and nothing else.

In [ ]:
csv_files = []

for i in range(len(all_files)):
    if --:
        --

n_files = len(csv_files)

print(n_files, 'CSV files;', len(all_files) - n_files, 'other file(s) skipped')
print('The 24 water files and nothing else?', n_files == 24 and csv_files[0] == 'water_AHG.csv' and csv_files[-1] == 'water_EPS.csv')

What would `pd.read_csv` do with `water_notes.txt` if the loop in Question 2 reached it? (The tutorial's section 6 says.) Why is it better to filter the names first than to let that happen?

*Your answer:*

**Question 2 — One number per building**

Loop over `csv_files`. For each file: read it, take the building code from its name, remove the outliers with the 3 standard deviation rule (the limits are written for you), and then **append** three things to three lists:

- `buildings`: the building code of each file
- `n_removed`: how many days the rule removed from that file (rows before minus rows after)
- `mean_gallons`: the mean of the `Water ( Gallons )` column **after** the outliers are removed

The last line turns the three lists into one table, `summary_df`, with one row per building. The check lines confirm that it has one row per CSV file and that DCP's row matches the tutorial (57 days removed).

In [ ]:
buildings = []
n_removed = []
mean_gallons = []

for --:
    file_name = csv_files[i]
    water_df = pd.read_csv(--)
    building = --

    w_mean = np.mean(water_df[col])
    w_std = np.std(water_df[col], ddof=1)
    upper = w_mean + 3 * w_std
    lower = w_mean - 3 * w_std
    n_before = len(water_df)
    water_df = water_df[(water_df[col] < upper) & (water_df[col] > lower)]

    # Add this building's code, its number of removed days, and its cleaned mean to the three lists
    --
    --
    --

summary_df = pd.DataFrame({'building': buildings, 'n_removed': n_removed, 'mean_gallons': mean_gallons})

print(len(summary_df), 'rows - one per CSV file?', len(summary_df) == n_files)
print('DCP matches the tutorial (57 removed)?', summary_df.loc[summary_df['building'] == 'DCP', 'n_removed'].item() == 57)
summary_df

BMA loses the most days to the rule, and its mean changes more than any other building's. Look at its row, then at `pd.read_csv(DATA_DIR / 'water_BMA.csv')[col].describe()` in the scratch cell at the end. What does the raw data look like, and would you trust the 3 standard deviation rule on it?

*Your answer:*

**Question 3 — Which building, and a picture of all 24**

Find the building with the **highest** cleaned mean with Lab 7's "largest so far" loop over the `means` array (the `mean_gallons` column as a NumPy array), remembering the building code along with the value. Then add up the removed days, and draw one bar per building.

- `top_building`, `top_mean`: the building with the highest cleaned mean, and that mean
- `total_removed`: the removed days of all 24 buildings added together
- `fig`: a bar chart, one bar per building, with the cleaned mean as its height, with the labels given

The check line compares your loop with pandas and your total with Python's `sum`.

In [ ]:
means = summary_df['mean_gallons'].to_numpy()

top_mean = means[0]
top_building = buildings[0]

for i in range(len(means)):
    if --:
        top_mean = --
        top_building = --

total_removed = --

print('Highest mean after cleaning:', top_building, round(top_mean, 1), 'gallons per day')
print(total_removed, 'outlier days removed across', n_files, 'buildings')
print('Matches pandas?', top_building == summary_df['building'][summary_df['mean_gallons'].idxmax()], total_removed == sum(n_removed))

fig, ax = plt.subplots(figsize=(10, 4))
# One bar per building: the codes along the x axis, the cleaned means as heights
--
ax.set_xlabel('Building')
ax.set_ylabel('Mean water consumption (gallons per day)')
ax.set_title('Campus buildings, outlier days removed')
fig.savefig(FIGURES_DIR / 'lab09_building_means.png', dpi=150)

Compute the mean of each building **before** outlier removal (a second, shorter loop in the scratch cell) and find the top building that way. Is it the same building? What single thing in that building's file makes the difference?

*Your answer:*

**Quiz scratch space**

The Canvas quiz asks about the values above. Use the cell below during the quiz.

**DO NOT OVERWRITE ANY VARIABLES ABOVE THIS CELL. CREATE NEW VARIABLES AS NEEDED.**

In [ ]:
# Use this cell for quiz calculations

## Wrap-up

**Runtime > Restart session and run all**, confirm every cell runs top to bottom, then **File > Download > Download .ipynb** and upload it to Gradescope for feedback. The submission is not graded. Keep this notebook open for the Canvas quiz.